# CCLE TP53 Project EDA: mRNA-first dataset preparation

This notebook prepares the CCLE dataset for an **mRNA-first TP53 mutation prediction project**.

The professor's guidance is built into the notebook design:
- **Primary signal source:** mRNA expression
- **Secondary covariates for fine-tuning only:** lineage, sex, and optionally disease
- **Primary modeling label:** `TP53_mut_nonsyn`

The notebook therefore produces three aligned outputs for the modeling notebook:
1. **Filtered RNA expression matrix**
2. **Small metadata table** for optional fine-tuning
3. **Labels and split membership**

Important modeling rule: any **supervised gene selection used for the final models must happen inside the modeling pipeline**, not here in EDA.


## 0. Setup

This section defines paths, configuration, and helper functions for:
- TP53 label construction
- train-only RNA filtering
- exploratory RNA association analysis
- sample QC / outlier flagging
- quick metadata-only baselines


In [1]:
from pathlib import Path
from IPython.display import display

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scipy
from scipy import stats
import sklearn
import statsmodels

from statsmodels.stats.multitest import multipletests
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

RNG = 42
DATA_DIR = Path("data")
OUT_DIR = Path("processed")
OUT_DIR.mkdir(exist_ok=True, parents=True)

MODEL_PATH = DATA_DIR / "Model.csv"
MUT_PATH = DATA_DIR / "OmicsSomaticMutations.csv"
EXPR_PATH = DATA_DIR / "OmicsExpressionProteinCodingGenesTPMLogp1.csv"

CONFIG = {
    "primary_label": "TP53_mut_nonsyn",
    "detect_expr_threshold": 0.10,
    "min_detect_frac": 0.05,
    "min_variance": 1e-4,
    "corr_alpha": 0.05,
    "corr_min_abs_r": 0.15,
    "max_corr_features_for_eda": 300,
    "outlier_vote_possible": 3,
    "outlier_vote_sure": 4,
    "outlier_contamination": 0.05,
    "min_meta_category_n": 15,
    "test_size": 0.15,
    "val_size_within_temp": 0.15 / 0.85,
}

for path in [MODEL_PATH, MUT_PATH, EXPR_PATH]:
    assert path.exists(), f"Missing file: {path}"


def choose_first_present(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


def collapse_duplicate_gene_columns(df):
    n_dup = int(df.columns.duplicated().sum())
    if n_dup == 0:
        return df, 0
    collapsed = df.T.groupby(level=0).mean().T
    return collapsed, n_dup


def robust_zscore(values):
    s = pd.Series(values, copy=True)
    med = s.median()
    mad = np.median(np.abs(s - med))
    if mad == 0 or pd.isna(mad):
        return pd.Series(0.0, index=s.index)
    return 0.6745 * (s - med) / mad


def mad_flag(values, threshold=3.5):
    return robust_zscore(values).abs() > threshold


def make_dense_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


TP53_TEXT_COLS = [
    "VariantClassification",
    "VariantInfo",
    "VariantType",
    "ProteinChange",
    "HGVSp_Short",
    "HGVSp",
    "RefSeq",
]


def classify_tp53_event(row):
    vc = str(row.get("VariantClassification", "")).strip().lower()
    vt = str(row.get("VariantType", "")).strip().lower()
    text = " ".join(str(row.get(c, "")) for c in TP53_TEXT_COLS).lower()

    if vc in {
        "frame_shift_del", "frame_shift_ins", "frameshift_deletion", "frameshift_insertion",
        "nonsense_mutation", "splice_site", "translation_start_site",
        "start_codon_del", "start_codon_ins", "stop_codon_del", "stop_codon_ins",
    }:
        return "lof_or_splice"

    if vc in {"missense_mutation", "in_frame_del", "in_frame_ins"}:
        return "missense_or_inframe"

    if vc in {"silent", "synonymous"}:
        return "synonymous"

    lof_terms = [
        "frameshift", "frame_shift", "nonsense", "stop_gained", "stopgain",
        "splice", "start_lost", "stop_lost", "trunc", "truncating",
    ]
    missense_terms = [
        "missense", "nonsynonymous", "non_synonymous", "inframe", "in_frame",
        "nonframeshift", "non_frameshift",
    ]
    syn_terms = ["synonymous", "silent"]

    if any(term in text for term in lof_terms):
        return "lof_or_splice"
    if any(term in text for term in missense_terms):
        return "missense_or_inframe"
    if any(term in text for term in syn_terms):
        return "synonymous"
    if vc:
        return "other_or_unknown"
    if vt in {"snp", "dnp", "tnp", "onp", "del", "ins"}:
        return "other_or_unknown"
    return "unknown"


def corr_against_binary(X, y, method="pearson"):
    assert X.index.equals(y.index), "X and y must have aligned indices."
    y = pd.Series(y, index=X.index).astype(float)
    y_vec = y.to_numpy()
    cols = X.columns

    if method == "spearman":
        X_use = X.rank(axis=0, method="average").to_numpy(dtype=float)
        y_use = pd.Series(y_vec, index=y.index).rank(method="average").to_numpy(dtype=float)
    else:
        X_use = X.to_numpy(dtype=float)
        y_use = y_vec.astype(float)

    x_std = X_use.std(axis=0, ddof=1)
    valid = x_std > 0
    valid_cols = cols[valid]

    out = pd.DataFrame(index=cols)
    out[f"{method}_r"] = np.nan
    out[f"{method}_p"] = np.nan
    out[f"{method}_p_bh"] = np.nan

    if valid.sum() == 0:
        return out

    Xv = X_use[:, valid]
    yc = y_use - y_use.mean()
    Xc = Xv - Xv.mean(axis=0, keepdims=True)

    denom = np.sqrt((Xc ** 2).sum(axis=0) * (yc ** 2).sum())
    r = (Xc * yc[:, None]).sum(axis=0) / np.clip(denom, 1e-12, None)

    n = len(y_use)
    t_stat = r * np.sqrt((n - 2) / np.clip(1 - r ** 2, 1e-12, None))
    p = 2 * stats.t.sf(np.abs(t_stat), df=n - 2)
    p_bh = multipletests(p, method="fdr_bh")[1]

    out.loc[valid_cols, f"{method}_r"] = r
    out.loc[valid_cols, f"{method}_p"] = p
    out.loc[valid_cols, f"{method}_p_bh"] = p_bh
    return out


def collapse_categories(train_s, val_s, min_n=15, other_label="Other"):
    train_s = train_s.fillna("Unknown").astype(str)
    val_s = val_s.fillna("Unknown").astype(str)
    counts = train_s.value_counts()
    major = set(counts[counts >= min_n].index.tolist())
    train_out = train_s.where(train_s.isin(major), other=other_label)
    val_out = val_s.where(val_s.isin(major), other=other_label)
    return train_out, val_out, counts


def eval_meta_baseline(train_df, val_df, y_train, y_val, cols, min_n=15):
    if len(cols) == 0:
        return None

    train_proc = pd.DataFrame(index=train_df.index)
    val_proc = pd.DataFrame(index=val_df.index)

    for c in cols:
        train_proc[c], val_proc[c], _ = collapse_categories(train_df[c], val_df[c], min_n=min_n)

    enc = make_dense_onehot_encoder()
    Xtr = enc.fit_transform(train_proc)
    Xva = enc.transform(val_proc)

    clf = LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RNG)
    clf.fit(Xtr, y_train)
    prob = clf.predict_proba(Xva)[:, 1]

    return {
        "model": "+".join(cols),
        "n_features": Xtr.shape[1],
        "auroc": roc_auc_score(y_val, prob),
        "auprc": average_precision_score(y_val, prob),
    }


def save_parquet_with_id(df, filename, id_col="ModelID"):
    path = OUT_DIR / filename
    out = df.copy()
    out.insert(0, id_col, out.index.astype(str))
    out.to_parquet(path, index=False)
    return path


def save_series_with_id(s, filename, value_name=None, id_col="ModelID"):
    path = OUT_DIR / filename
    s = s.copy()

    # Ensure the value column has a valid string name
    if value_name is None:
        if s.name is None:
            value_name = filename.replace(".parquet", "")
        else:
            value_name = str(s.name)
    else:
        value_name = str(value_name)

    # Ensure the index column name is also a string
    if id_col is None:
        id_col = s.index.name if s.index.name is not None else "ModelID"
    id_col = str(id_col)

    df = s.rename(value_name).rename_axis(id_col).reset_index()

    # Final safety: force all column names to strings
    df.columns = [str(c) for c in df.columns]

    df.to_parquet(path, index=False)
    return path

print(
    f"pandas {pd.__version__} | numpy {np.__version__} | scipy {scipy.__version__} | "
    f"sklearn {sklearn.__version__} | statsmodels {statsmodels.__version__}"
)
print(json.dumps(CONFIG, indent=2))


pandas 3.0.2 | numpy 2.4.4 | scipy 1.17.1 | sklearn 1.8.0 | statsmodels 0.14.6
{
  "primary_label": "TP53_mut_nonsyn",
  "detect_expr_threshold": 0.1,
  "min_detect_frac": 0.05,
  "min_variance": 0.0001,
  "corr_alpha": 0.05,
  "corr_min_abs_r": 0.15,
  "max_corr_features_for_eda": 300,
  "outlier_vote_possible": 3,
  "outlier_vote_sure": 4,
  "outlier_contamination": 0.05,
  "min_meta_category_n": 15,
  "test_size": 0.15,
  "val_size_within_temp": 0.17647058823529413
}


## 1. Load data, keep small metadata, and build TP53 labels

The goal here is to build a **clean sample table** with:
- expression data
- a small metadata subset for optional fine-tuning
- TP53 labels

The metadata table is intentionally small. It is **not** meant to replace the mRNA signal.


In [2]:
model = pd.read_csv(MODEL_PATH)
muts = pd.read_csv(MUT_PATH, low_memory=False)
expr = pd.read_csv(EXPR_PATH, index_col=0)

assert "ModelID" in muts.columns, "Expected ModelID in mutation file."
assert "HugoSymbol" in muts.columns, "Expected HugoSymbol in mutation file."

expr.index.name = "ModelID"
expr.columns = expr.columns.str.replace(r"\s*\([^)]*\)$", "", regex=True)
expr, n_collapsed_gene_names = collapse_duplicate_gene_columns(expr)

lineage_col = choose_first_present(
    model.columns,
    ["OncotreeLineage", "OncotreeLineageDerived", "Lineage", "lineage"],
)
disease_col = choose_first_present(
    model.columns,
    ["OncotreePrimaryDisease", "PrimaryDisease", "Primary Disease", "Disease"],
)
sex_col = choose_first_present(
    model.columns,
    ["Sex", "sex", "Gender", "gender", "PatientSex"],
)
name_col = choose_first_present(
    model.columns,
    ["StrippedCellLineName", "CellLineName", "ModelName", "CCLEName"],
)

model_keep = ["ModelID"] + [c for c in [name_col, lineage_col, sex_col, disease_col] if c is not None]
model_meta = model[model_keep].drop_duplicates("ModelID").set_index("ModelID")

expr_ids = pd.Index(expr.index.unique(), name="ModelID")
mut_ids = pd.Index(pd.Series(muts["ModelID"].dropna().astype(str).unique()).sort_values(), name="ModelID")
model_ids = pd.Index(model_meta.index.unique().astype(str), name="ModelID")

overlap_summary = pd.Series(
    {
        "expression_samples": len(expr_ids),
        "mutation_covered_samples": len(mut_ids),
        "metadata_samples": len(model_ids),
        "expr_and_mut_overlap": len(expr_ids.intersection(mut_ids)),
        "expr_and_meta_overlap": len(expr_ids.intersection(model_ids)),
        "mut_and_meta_overlap": len(mut_ids.intersection(model_ids)),
        "expr_mut_meta_overlap": len(expr_ids.intersection(mut_ids).intersection(model_ids)),
    },
    name="n",
)

print(f"Model:      {model.shape}")
print(f"Mutations:  {muts.shape}")
print(f"Expression: {expr.shape}")
print(f"Collapsed duplicate gene symbols after stripping suffixes: {n_collapsed_gene_names}")
print(f"Metadata columns kept: {[c for c in [name_col, lineage_col, sex_col, disease_col] if c is not None]}")
display(overlap_summary.to_frame())

tp53_muts = muts.loc[muts["HugoSymbol"].eq("TP53")].copy()
tp53_muts["tp53_event_class"] = tp53_muts.apply(classify_tp53_event, axis=1)

priority_map = {
    "lof_or_splice": 5,
    "missense_or_inframe": 4,
    "other_or_unknown": 3,
    "synonymous": 2,
    "unknown": 1,
}
tp53_muts["priority"] = tp53_muts["tp53_event_class"].map(priority_map).fillna(0).astype(int)

best_tp53 = (
    tp53_muts.sort_values(["ModelID", "priority"], ascending=[True, False])
    .drop_duplicates("ModelID", keep="first")
    .set_index("ModelID")
)

called_ids = set(muts["ModelID"].dropna().astype(str).unique())
expr_ids = set(expr.index.astype(str))
common = sorted(called_ids & expr_ids)

expr.index = expr.index.astype(str)
X_all = expr.loc[common].copy()

sample_meta = pd.DataFrame(index=pd.Index(common, name="ModelID")).join(model_meta, how="left")
sample_meta["TP53_event_class"] = best_tp53["tp53_event_class"].reindex(common)
sample_meta["tp53_event_count"] = tp53_muts.groupby("ModelID").size().reindex(common).fillna(0).astype(int)

sample_meta["TP53_mut_any"] = sample_meta.index.isin(best_tp53.index.astype(str)).astype(int)
sample_meta["TP53_mut_nonsyn"] = sample_meta["TP53_event_class"].isin(["lof_or_splice", "missense_or_inframe"]).astype(int)
sample_meta["TP53_mut_lof"] = sample_meta["TP53_event_class"].eq("lof_or_splice").astype(int)
sample_meta["TP53_mut_ambiguous"] = sample_meta["TP53_event_class"].isin(["other_or_unknown", "unknown", "synonymous"]).astype(int)

mut_type = sample_meta["TP53_event_class"].fillna("wild_type").rename("mut_type")
y_primary = sample_meta[CONFIG["primary_label"]].rename(CONFIG["primary_label"])

print(f"Usable samples with mutation-call coverage and expression: {len(common):,}")
print(f"Primary label ({CONFIG['primary_label']}): {int(y_primary.sum()):,} positives ({y_primary.mean():.1%})")
print(f"TP53_mut_any:   {int(sample_meta['TP53_mut_any'].sum()):,} positives ({sample_meta['TP53_mut_any'].mean():.1%})")
print(f"TP53_mut_lof:   {int(sample_meta['TP53_mut_lof'].sum()):,} positives ({sample_meta['TP53_mut_lof'].mean():.1%})")
print(f"Cell lines with >1 TP53 event: {(sample_meta['tp53_event_count'] > 1).sum():,}")

event_counts = (
    sample_meta["TP53_event_class"]
    .fillna("wild_type")
    .value_counts(dropna=False)
    .rename_axis("event_class")
    .to_frame("n_cell_lines")
)
display(event_counts)


Model:      (2116, 49)
Mutations:  (751310, 70)
Expression: (1684, 19205)
Collapsed duplicate gene symbols after stripping suffixes: 0
Metadata columns kept: ['StrippedCellLineName', 'OncotreeLineage', 'Sex', 'OncotreePrimaryDisease']


,n
expression_samples,1684
mutation_covered_samples,1939
metadata_samples,2116
expr_and_mut_overlap,1608
expr_and_meta_overlap,1684
mut_and_meta_overlap,1939
expr_mut_meta_overlap,1608


Usable samples with mutation-call coverage and expression: 1,608
Primary label (TP53_mut_nonsyn): 987 positives (61.4%)
TP53_mut_any:   987 positives (61.4%)
TP53_mut_lof:   362 positives (22.5%)
Cell lines with >1 TP53 event: 115


,n_cell_lines
event_class,
missense_or_inframe,625
wild_type,621
lof_or_splice,362


## 1b. Inspect raw TP53 mutation annotations

This block directly inspects the raw TP53 mutation rows using the mutation-annotation columns most relevant for manual review:

- `HugoSymbol`
- `VariantType`
- `VariantInfo`
- `DNAChange`
- `ProteinChange`

This does **not** replace the derived label logic above. It is a QC step to verify that the raw TP53 rows look biologically and syntactically sensible before continuing.


In [3]:
# Direct TP53 / p53 mutation annotation QC
required_tp53_cols = [
    "HugoSymbol",
    "VariantType",
    "VariantInfo",
    "DNAChange",
    "ProteinChange",
]

missing_tp53_cols = [c for c in required_tp53_cols if c not in tp53_muts.columns]
if missing_tp53_cols:
    print(f"Missing expected TP53 mutation columns: {missing_tp53_cols}")

tp53_view_cols = [c for c in required_tp53_cols if c in tp53_muts.columns]
tp53_view = tp53_muts[tp53_view_cols].copy()

print(f"Total TP53 mutation rows: {len(tp53_view):,}")
print(f"Unique ModelID with any TP53 row: {tp53_muts['ModelID'].astype(str).nunique():,}")

print("First TP53 rows (raw annotations):")
display(tp53_view.head(20))

if "VariantType" in tp53_view.columns:
    print("VariantType counts:")
    display(tp53_view["VariantType"].fillna("<NA>").value_counts(dropna=False).to_frame("n_rows"))

if "VariantInfo" in tp53_view.columns:
    print("VariantInfo counts:")
    display(tp53_view["VariantInfo"].fillna("<NA>").value_counts(dropna=False).to_frame("n_rows"))

if "ProteinChange" in tp53_view.columns:
    print("Top ProteinChange values:")
    display(tp53_view["ProteinChange"].fillna("<NA>").value_counts(dropna=False).head(25).to_frame("n_rows"))

missingness_summary = pd.Series(
    {c: int(tp53_view[c].isna().sum()) for c in tp53_view.columns},
    name="n_missing",
)
print("Missing values across TP53 review columns:")
display(missingness_summary.to_frame())

# Helpful QC views for rows that may need manual inspection
qc_examples = {}
if "ProteinChange" in tp53_view.columns:
    qc_examples["missing_protein_change"] = tp53_muts.loc[
        tp53_muts["ProteinChange"].isna(),
        ["ModelID"] + tp53_view_cols + [c for c in ["VariantClassification", "tp53_event_class"] if c in tp53_muts.columns],
    ].head(15)

if "DNAChange" in tp53_view.columns:
    qc_examples["missing_dna_change"] = tp53_muts.loc[
        tp53_muts["DNAChange"].isna(),
        ["ModelID"] + tp53_view_cols + [c for c in ["VariantClassification", "tp53_event_class"] if c in tp53_muts.columns],
    ].head(15)

if "tp53_event_class" in tp53_muts.columns:
    qc_examples["ambiguous_or_unknown"] = tp53_muts.loc[
        tp53_muts["tp53_event_class"].isin(["other_or_unknown", "unknown", "synonymous"]),
        ["ModelID"] + tp53_view_cols + [c for c in ["VariantClassification", "tp53_event_class"] if c in tp53_muts.columns],
    ].head(20)

for name, df in qc_examples.items():
    if len(df) > 0:
        print(f"QC example: {name}")
        display(df)

# Compact table to compare raw mutation annotations against the derived TP53 event label
tp53_review_summary = tp53_muts.loc[:, [c for c in [
    "ModelID", "HugoSymbol", "VariantType", "VariantInfo", "DNAChange", "ProteinChange",
    "VariantClassification", "tp53_event_class"
] if c in tp53_muts.columns]].copy()

display(tp53_review_summary.head(20))


Total TP53 mutation rows: 1,319
Unique ModelID with any TP53 row: 1,174
First TP53 rows (raw annotations):


,HugoSymbol,VariantType,VariantInfo,DNAChange,ProteinChange
154890,TP53,SNV,missense_variant,ENST00000269305.9:c.1165G>T,p.G389W
154891,TP53,deletion,frameshift_variant,ENST00000269305.9:c.1146del,p.K382NfsTer40
154892,TP53,SNV,splice_acceptor_variant,ENST00000269305.9:c.1101-2A>C,NaN
154893,TP53,SNV,splice_acceptor_variant,ENST00000269305.9:c.1101-2A>C,NaN
154894,TP53,deletion,splice_donor_variant&coding_sequence_variant,ENST00000269305.9:c.1097_1100+1del,NaN
154895,TP53,substitution,missense_variant,ENST00000269305.9:c.1049_1050delinsCT,p.L350P
154896,TP53,SNV,missense_variant,ENST00000269305.9:c.1049T>C,p.L350P
154897,TP53,SNV,stop_gained,ENST00000269305.9:c.1045G>T,p.E349Ter
154898,TP53,SNV,missense_variant,ENST00000269305.9:c.1040C>T,p.A347V
154899,TP53,SNV,missense_variant,ENST00000269305.9:c.1039G>C,p.A347P


VariantType counts:


,n_rows
VariantType,
SNV,1106
deletion,147
insertion,43
substitution,23


VariantInfo counts:


...

Missing values across TP53 review columns:


,n_missing
HugoSymbol,0
VariantType,0
VariantInfo,0
DNAChange,0
ProteinChange,111


QC example: missing_protein_change


,ModelID,HugoSymbol,VariantType,VariantInfo,DNAChange,ProteinChange,tp53_event_class
154892,ACH-000997,TP53,SNV,splice_acceptor_variant,ENST00000269305.9:c.1101-2A>C,NaN,lof_or_splice
154893,ACH-001061,TP53,SNV,splice_acceptor_variant,ENST00000269305.9:c.1101-2A>C,NaN,lof_or_splice
154894,ACH-000227,TP53,deletion,splice_donor_variant&coding_sequence_variant,ENST00000269305.9:c.1097_1100+1del,NaN,lof_or_splice
154919,ACH-001143,TP53,SNV,splice_acceptor_variant,ENST00000269305.9:c.994-1G>A,NaN,lof_or_splice
154920,ACH-000484,TP53,SNV,splice_acceptor_variant,ENST00000269305.9:c.994-2A>G,NaN,lof_or_splice
154921,ACH-000243,TP53,deletion,splice_donor_variant&splice_donor_5th_base_var...,ENST00000269305.9:c.972_993+16del,NaN,lof_or_splice
154922,ACH-000557,TP53,SNV,splice_donor_variant,ENST00000269305.9:c.993+2T>G,NaN,lof_or_splice
154923,ACH-000502,TP53,SNV,splice_donor_variant,ENST00000269305.9:c.993+1G>A,NaN,lof_or_splice
154924,ACH-002665,TP53,SNV,splice_donor_variant,ENST00000269305.9:c.993+1G>C,NaN,lof_or_splice
154925,ACH-000021,TP53,SNV,splice_donor_variant,ENST00000269305.9:c.993+1G>T,NaN,lof_or_splice


,ModelID,HugoSymbol,VariantType,VariantInfo,DNAChange,ProteinChange,tp53_event_class
154890,ACH-000940,TP53,SNV,missense_variant,ENST00000269305.9:c.1165G>T,p.G389W,missense_or_inframe
154891,ACH-000632,TP53,deletion,frameshift_variant,ENST00000269305.9:c.1146del,p.K382NfsTer40,lof_or_splice
154892,ACH-000997,TP53,SNV,splice_acceptor_variant,ENST00000269305.9:c.1101-2A>C,NaN,lof_or_splice
154893,ACH-001061,TP53,SNV,splice_acceptor_variant,ENST00000269305.9:c.1101-2A>C,NaN,lof_or_splice
154894,ACH-000227,TP53,deletion,splice_donor_variant&coding_sequence_variant,ENST00000269305.9:c.1097_1100+1del,NaN,lof_or_splice
154895,ACH-001975,TP53,substitution,missense_variant,ENST00000269305.9:c.1049_1050delinsCT,p.L350P,missense_or_inframe
154896,ACH-002283,TP53,SNV,missense_variant,ENST00000269305.9:c.1049T>C,p.L350P,missense_or_inframe
154897,ACH-000720,TP53,SNV,stop_gained,ENST00000269305.9:c.1045G>T,p.E349Ter,lof_or_splice
154898,ACH-000701,TP53,SNV,missense_variant,ENST00000269305.9:c.1040C>T,p.A347V,missense_or_inframe
154899,ACH-000803,TP53,SNV,missense_variant,ENST00000269305.9:c.1039G>C,p.A347P,missense_or_inframe


## 2. Train / validation / test split

The split is stratified on the **primary label** `TP53_mut_nonsyn`.

This keeps the EDA and modeling notebooks aligned from the start.


In [4]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X_all,
    y_primary,
    test_size=CONFIG["test_size"],
    stratify=y_primary,
    random_state=RNG,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=CONFIG["val_size_within_temp"],
    stratify=y_temp,
    random_state=RNG,
)

split_membership = pd.Series(index=X_all.index, dtype="object")
split_membership.loc[X_train.index] = "train"
split_membership.loc[X_val.index] = "val"
split_membership.loc[X_test.index] = "test"
sample_meta["split"] = split_membership

split_summary = pd.DataFrame(
    {
        "n": [len(y_train), len(y_val), len(y_test)],
        "tp53_mut_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
    },
    index=["train", "val", "test"],
)
display(split_summary.round(3))


,n,tp53_mut_rate
train,1124,0.614
val,242,0.612
test,242,0.616


## 3. Metadata diagnostics for optional fine-tuning variables

These checks are here because metadata is allowed for **fine-tuning**, but it is not the main modeling source.

We therefore inspect:
- missingness
- category balance across splits
- primary-label rate by category in training


In [5]:
present_meta_cols = [c for c in [lineage_col, sex_col, disease_col] if c is not None]
recommended_meta_cols = [c for c in [lineage_col, sex_col] if c is not None]

if not present_meta_cols:
    print("No metadata covariates found beyond ModelID / names.")
else:
    meta_missing = pd.DataFrame(
        {
            "missing_n": sample_meta[present_meta_cols].isna().sum(),
            "missing_frac": sample_meta[present_meta_cols].isna().mean(),
        }
    ).sort_values("missing_frac", ascending=False)
    print("Metadata missingness:")
    display(meta_missing.round(3))

    for col in present_meta_cols:
        print(f"\n=== {col} ===")
        top_by_split = (
            sample_meta[["split", col]]
            .assign(**{col: lambda d: d[col].fillna("Unknown").astype(str)})
            .groupby(["split", col])
            .size()
            .rename("n")
            .reset_index()
        )
        display(top_by_split.sort_values(["split", "n"], ascending=[True, False]).groupby("split").head(8))

        train_rates = (
            sample_meta.loc[X_train.index, [col, CONFIG["primary_label"]]]
            .assign(**{col: lambda d: d[col].fillna("Unknown").astype(str)})
            .groupby(col)
            .agg(n=(CONFIG["primary_label"], "size"), tp53_rate=(CONFIG["primary_label"], "mean"))
            .sort_values("n", ascending=False)
        )
        display(train_rates.head(15).round(3))

print("Recommended metadata columns for the main modeling notebook:")
print(recommended_meta_cols if recommended_meta_cols else "[]")


Metadata missingness:


,missing_n,missing_frac
OncotreeLineage,5,0.003
Sex,0,0.000
OncotreePrimaryDisease,0,0.000



=== OncotreeLineage ===


,split,OncotreeLineage,n
13,test,Lung,22
21,test,Skin,21
3,test,Bowel,19
14,test,Lymphoid,18
5,test,CNS/Brain,14
10,test,Head and Neck,13
17,test,Pancreas,13
16,test,Ovary/Fallopian Tube,12
41,train,Lung,148
42,train,Lymphoid,128


,n,tp53_rate
OncotreeLineage,,
Lung,148,0.905
Lymphoid,128,0.484
Skin,77,0.325
CNS/Brain,68,0.647
Esophagus/Stomach,56,0.857
Bowel,54,0.796
Myeloid,50,0.580
Breast,50,0.800
Head and Neck,50,0.880


...

,n,tp53_rate
Sex,,
Male,604,0.629
Female,471,0.601
Unknown,49,0.551



=== OncotreePrimaryDisease ===


,split,OncotreePrimaryDisease,n
8,test,Colorectal Adenocarcinoma,19
29,test,Melanoma,19
36,test,Non-Small Cell Lung Cancer,14
18,test,Head and Neck Squamous Cell Carcinoma,12
39,test,Ovarian Epithelial Tumor,12
40,test,Pancreatic Adenocarcinoma,12
10,test,Diffuse Glioma,11
23,test,Invasive Breast Carcinoma,10
108,train,Non-Small Cell Lung Cancer,99
97,train,Melanoma,72


,n,tp53_rate
OncotreePrimaryDisease,,
Non-Small Cell Lung Cancer,99,0.879
Melanoma,72,0.292
Mature B-Cell Neoplasms,71,0.507
Diffuse Glioma,56,0.714
Colorectal Adenocarcinoma,52,0.808
Head and Neck Squamous Cell Carcinoma,48,0.917
Lung Neuroendocrine Tumor,46,0.957
Invasive Breast Carcinoma,44,0.818
Ovarian Epithelial Tumor,40,0.700


Recommended metadata columns for the main modeling notebook:
['OncotreeLineage', 'Sex']


## 4. Training-only low-information RNA filtering

This is the core RNA preprocessing step used for downstream modeling.

The key rule is preserved:
- filtering thresholds are learned on the **training split only**
- the same kept-gene mask is then applied to validation and test


In [6]:
detected_frac = (X_train > CONFIG["detect_expr_threshold"]).mean(axis=0)
variance = X_train.var(axis=0, ddof=1)

keep_gene_mask = (
    (detected_frac >= CONFIG["min_detect_frac"])
    & (variance >= CONFIG["min_variance"])
)

gene_filter_qc = pd.DataFrame(
    {
        "detected_frac": detected_frac,
        "variance": variance,
        "fail_detect_frac": detected_frac < CONFIG["min_detect_frac"],
        "fail_variance": variance < CONFIG["min_variance"],
        "keep": keep_gene_mask,
    }
).sort_values(["keep", "variance", "detected_frac"], ascending=[False, False, False])

X_train_filt = X_train.loc[:, keep_gene_mask].copy()
X_val_filt = X_val.loc[:, keep_gene_mask].copy()
X_test_filt = X_test.loc[:, keep_gene_mask].copy()
X_rna_filtered_all = X_all.loc[:, keep_gene_mask].copy()

print(f"Genes before filtering: {X_train.shape[1]:,}")
print(f"Genes after filtering:  {X_train_filt.shape[1]:,}")
print(f"Removed low-information genes: {(~keep_gene_mask).sum():,}")

removal_summary = pd.Series(
    {
        "fail_detect_frac_only": int((gene_filter_qc["fail_detect_frac"] & ~gene_filter_qc["fail_variance"]).sum()),
        "fail_variance_only": int((gene_filter_qc["fail_variance"] & ~gene_filter_qc["fail_detect_frac"]).sum()),
        "fail_both": int((gene_filter_qc["fail_detect_frac"] & gene_filter_qc["fail_variance"]).sum()),
    },
    name="n_genes",
)
display(removal_summary.to_frame())


Genes before filtering: 19,205
Genes after filtering:  17,764
Removed low-information genes: 1,441


,n_genes
fail_detect_frac_only,1307
fail_variance_only,0
fail_both,134


## 5. Exploratory mRNA association analysis

This section is **EDA only**.

It is useful for understanding RNA structure and TP53-associated genes, but it is **not** the final supervised feature selection step for the main models.

The final models will redo supervised selection **inside cross-validation** in the modeling notebook.


In [7]:
corr_pearson = corr_against_binary(X_train_filt, y_train, method="pearson")
corr_spearman = corr_against_binary(X_train_filt, y_train, method="spearman")

corr = corr_pearson.join(corr_spearman)
corr["abs_pearson_r"] = corr["pearson_r"].abs()
corr["abs_spearman_r"] = corr["spearman_r"].abs()
corr = corr.sort_values("abs_pearson_r", ascending=False)

corr_ranked = corr.loc[
    (corr["pearson_p_bh"] <= CONFIG["corr_alpha"])
    & (corr["spearman_p_bh"] <= CONFIG["corr_alpha"])
    & (corr["abs_pearson_r"] >= CONFIG["corr_min_abs_r"])
].sort_values("abs_pearson_r", ascending=False)

TP53_TARGETS = {
    "APAF1", "ATF3", "BAK1", "BAX", "BBC3", "BID", "BTG2", "CASP6", "CCNG1", "CCNG2",
    "CDKN1A", "DDB2", "DRAM1", "EDA2R", "FAS", "FDXR", "FOXO3", "GADD45A", "GADD45B",
    "GDF15", "IGFBP3", "LIF", "LRDD", "MDM2", "MDM4", "NDRG1", "PERP", "PIDD1", "PLK2",
    "PLK3", "PMAIP1", "POLH", "PTEN", "RPS27L", "RRM2B", "RRAD", "SERPINE1", "SESN1",
    "SESN2", "SESN3", "SFN", "SIAH1", "THBS1", "TIGAR", "TNFRSF10B", "TP53AIP1",
    "TP53I3", "TP53INP1", "WIG1", "XPC", "ZMAT3",
}
TARGET_ALIASES = {"WIG1": "ZMAT3"}

targets_present = set()
for gene in sorted(TP53_TARGETS):
    if gene in X_train_filt.columns:
        targets_present.add(gene)
    elif gene in TARGET_ALIASES and TARGET_ALIASES[gene] in X_train_filt.columns:
        targets_present.add(TARGET_ALIASES[gene])

eda_target_overlap = pd.DataFrame(index=sorted(set(corr_ranked.head(CONFIG["max_corr_features_for_eda"]).index) | targets_present))
eda_target_overlap["from_corr_ranked"] = eda_target_overlap.index.isin(corr_ranked.head(CONFIG["max_corr_features_for_eda"]).index)
eda_target_overlap["from_tp53_target_list"] = eda_target_overlap.index.isin(targets_present)
eda_target_overlap["pearson_r"] = corr.reindex(eda_target_overlap.index)["pearson_r"]
eda_target_overlap["spearman_r"] = corr.reindex(eda_target_overlap.index)["spearman_r"]
eda_target_overlap["abs_pearson_r"] = eda_target_overlap["pearson_r"].abs()
eda_target_overlap = eda_target_overlap.sort_values("abs_pearson_r", ascending=False)

summary = pd.Series(
    {
        "genes_tested": int(corr["pearson_r"].notna().sum()),
        "pearson_bh_sig": int((corr["pearson_p_bh"] <= CONFIG["corr_alpha"]).sum()),
        "spearman_bh_sig": int((corr["spearman_p_bh"] <= CONFIG["corr_alpha"]).sum()),
        "corr_ranked_genes": len(corr_ranked),
        "tp53_targets_present": len(targets_present),
    },
    name="n_genes",
)
display(summary.to_frame())

print("Top positively associated genes (EDA only):")
display(corr.sort_values("pearson_r", ascending=False).head(15)[["pearson_r", "pearson_p_bh", "spearman_r", "spearman_p_bh"]].round(4))

print("Top negatively associated genes (EDA only):")
display(corr.sort_values("pearson_r", ascending=True).head(15)[["pearson_r", "pearson_p_bh", "spearman_r", "spearman_p_bh"]].round(4))


,n_genes
genes_tested,17764
pearson_bh_sig,8713
spearman_bh_sig,9624
corr_ranked_genes,2232
tp53_targets_present,49


Top positively associated genes (EDA only):


,pearson_r,pearson_p_bh,spearman_r,spearman_p_bh
EPCAM,0.3368,0.0,0.3504,0.0
NSUN7,0.3332,0.0,0.3327,0.0
CDC42BPG,0.3192,0.0,0.3335,0.0
CLDN7,0.3184,0.0,0.3341,0.0
CNKSR1,0.3181,0.0,0.3528,0.0
PLS1,0.3162,0.0,0.3194,0.0
SPINT2,0.3123,0.0,0.3128,0.0
RCE1,0.3111,0.0,0.3259,0.0
OVOL2,0.3106,0.0,0.3443,0.0
MAP3K9,0.3105,0.0,0.3093,0.0


Top negatively associated genes (EDA only):


,pearson_r,pearson_p_bh,spearman_r,spearman_p_bh
EDA2R,-0.6215,0.0,-0.5581,0.0
MDM2,-0.5366,0.0,-0.5616,0.0
RPS27L,-0.5278,0.0,-0.5343,0.0
ZMAT3,-0.5267,0.0,-0.5192,0.0
CDKN1A,-0.5188,0.0,-0.5508,0.0
RRM2B,-0.4187,0.0,-0.4141,0.0
SESN1,-0.4142,0.0,-0.4348,0.0
BAX,-0.4073,0.0,-0.4090,0.0
PTCHD4,-0.4038,0.0,-0.4366,0.0
FDXR,-0.3986,0.0,-0.4008,0.0


## 6. Candidate sample QC flags

These are retained as a **sensitivity-analysis aid**.

The notebook does **not** drop outliers by default. The modeling notebook can compare the primary model with and without sure outliers.


In [8]:
CONTAM = CONFIG["outlier_contamination"]

votes = pd.DataFrame(index=X_train_filt.index)
sample_qc = pd.DataFrame(index=X_train_filt.index)

Xs = StandardScaler().fit_transform(X_train_filt)

Z = np.abs(stats.zscore(X_train_filt, axis=0, nan_policy="omit"))
extreme_gene_burden = pd.Series((Z > 4.0).sum(axis=1), index=X_train_filt.index)
votes["extreme_gene_burden"] = (extreme_gene_burden >= extreme_gene_burden.quantile(1 - CONTAM)).astype(int)
sample_qc["extreme_gene_burden"] = extreme_gene_burden

n_pcs_outlier = min(20, X_train_filt.shape[0] - 1, X_train_filt.shape[1])
pcs_outlier = PCA(n_components=n_pcs_outlier, random_state=RNG).fit_transform(Xs)
pc_distance = pd.Series(np.sqrt((pcs_outlier ** 2).sum(axis=1)), index=X_train_filt.index)
votes["pca_distance"] = (pc_distance >= pc_distance.quantile(1 - CONTAM)).astype(int)
sample_qc["pca_distance"] = pc_distance

iso = IsolationForest(contamination=CONTAM, random_state=RNG, n_jobs=-1)
votes["iforest"] = (iso.fit_predict(Xs) == -1).astype(int)

lof = LocalOutlierFactor(n_neighbors=20, contamination=CONTAM)
votes["lof"] = (lof.fit_predict(Xs) == -1).astype(int)

total_expr = X_train_filt.sum(axis=1)
votes["total_expr_mad"] = mad_flag(total_expr).astype(int)
sample_qc["total_expr"] = total_expr

n_detected = (X_train_filt > CONFIG["detect_expr_threshold"]).sum(axis=1)
votes["n_detected_mad"] = mad_flag(n_detected).astype(int)
sample_qc["n_detected"] = n_detected

votes["n_votes"] = votes.sum(axis=1)

sure_outliers = votes.index[votes["n_votes"] >= CONFIG["outlier_vote_sure"]]
possible_outliers = votes.index[votes["n_votes"] == CONFIG["outlier_vote_possible"]]

print("Fraction flagged by each detector:")
display(votes.drop(columns="n_votes").mean().round(3).to_frame("fraction_flagged"))
print("Samples by vote count:")
display(votes["n_votes"].value_counts().sort_index().to_frame("n_samples"))
print(f"Sure outliers (>={CONFIG['outlier_vote_sure']} votes): {len(sure_outliers)}")
print(f"Possible outliers (= {CONFIG['outlier_vote_possible']} votes): {len(possible_outliers)}")


Fraction flagged by each detector:


,fraction_flagged
extreme_gene_burden,0.052
pca_distance,0.051
iforest,0.051
lof,0.051
total_expr_mad,0.015
n_detected_mad,0.004


Samples by vote count:


,n_samples
n_votes,
0,976
1,86
2,33
3,20
4,7
5,2


Sure outliers (>=4 votes): 9
Possible outliers (= 3 votes): 20


## 7. Quick metadata-only validation benchmark

This is not the main model. It is a quick check of how much signal is already available in metadata alone.

That helps the modeling notebook answer a clean comparison:
- metadata-only
- mRNA-only
- mRNA + metadata


In [9]:
meta_baseline_rows = []

if lineage_col is not None:
    row = eval_meta_baseline(
        sample_meta.loc[X_train.index],
        sample_meta.loc[X_val.index],
        y_train,
        y_val,
        cols=[lineage_col],
        min_n=CONFIG["min_meta_category_n"],
    )
    if row is not None:
        meta_baseline_rows.append(row)

if sex_col is not None:
    row = eval_meta_baseline(
        sample_meta.loc[X_train.index],
        sample_meta.loc[X_val.index],
        y_train,
        y_val,
        cols=[c for c in [lineage_col, sex_col] if c is not None],
        min_n=CONFIG["min_meta_category_n"],
    )
    if row is not None:
        meta_baseline_rows.append(row)

if disease_col is not None and lineage_col is not None:
    row = eval_meta_baseline(
        sample_meta.loc[X_train.index],
        sample_meta.loc[X_val.index],
        y_train,
        y_val,
        cols=[lineage_col, disease_col],
        min_n=CONFIG["min_meta_category_n"],
    )
    if row is not None:
        row["note"] = "exploratory because disease may overlap strongly with lineage"
        meta_baseline_rows.append(row)

meta_baseline_df = pd.DataFrame(meta_baseline_rows)
if len(meta_baseline_df) == 0:
    print("No metadata-only baselines could be built.")
else:
    display(meta_baseline_df.sort_values("auprc", ascending=False).round(4))


,model,n_features,auroc,auprc,note
2,OncotreeLineage+OncotreePrimaryDisease,45,0.7148,0.7707,exploratory because disease may overlap strong...
0,OncotreeLineage,21,0.6766,0.7260,NaN
1,OncotreeLineage+Sex,24,0.6727,0.7236,NaN


## 8. Save artifacts for the mRNA-first modeling notebook

The official downstream artifacts are now:
- **filtered RNA matrix**
- **small metadata matrix**
- **labels + split membership**
- **EDA-only diagnostics** such as outliers and gene-filter QC

Unlike the earlier version, the notebook does **not** treat globally selected genes or PCA features as the official modeling inputs.


In [10]:
artifacts_written = []

X_meta = sample_meta[[c for c in [lineage_col, sex_col, disease_col] if c is not None]].copy()

schema = {
    "primary_label": CONFIG["primary_label"],
    "alternate_labels": ["TP53_mut_any", "TP53_mut_nonsyn", "TP53_mut_lof"],
    "lineage_col": lineage_col,
    "sex_col": sex_col,
    "disease_col": disease_col,
    "name_col": name_col,
    "available_meta_cols": [c for c in [lineage_col, sex_col, disease_col] if c is not None],
    "recommended_meta_cols": recommended_meta_cols,
    "detect_expr_threshold": CONFIG["detect_expr_threshold"],
    "min_detect_frac": CONFIG["min_detect_frac"],
    "min_variance": CONFIG["min_variance"],
    "corr_alpha": CONFIG["corr_alpha"],
    "corr_min_abs_r": CONFIG["corr_min_abs_r"],
    "max_corr_features_for_eda": CONFIG["max_corr_features_for_eda"],
    "split_sizes": split_summary["n"].to_dict(),
}

for obj, filename, mode in [
    (sample_meta, "ccle_sample_metadata.parquet", "df"),
    (X_rna_filtered_all, "ccle_X_rna_filtered_all.parquet", "df"),
    (X_meta, "ccle_X_meta.parquet", "df"),
    (gene_filter_qc, "ccle_gene_filter_qc.parquet", "df"),
    (corr, "ccle_corr_train_filtered_primary.parquet", "df"),
    (eda_target_overlap, "ccle_eda_target_overlap.parquet", "df"),
    (votes, "ccle_outlier_votes.parquet", "df"),
    (sample_qc, "ccle_sample_qc_metrics.parquet", "df"),
    (split_membership, "ccle_split_membership.parquet", "series"),
    (y_primary, "ccle_y_primary.parquet", "series"),
    (sample_meta["TP53_mut_any"], "ccle_y_all_any.parquet", "series"),
    (sample_meta["TP53_mut_nonsyn"], "ccle_y_all_nonsyn.parquet", "series"),
    (sample_meta["TP53_mut_lof"], "ccle_y_all_lof.parquet", "series"),
    (mut_type, "ccle_mut_type.parquet", "series"),
]:
    if mode == "df":
        artifacts_written.append(save_parquet_with_id(obj, filename))
    else:
        artifacts_written.append(save_series_with_id(obj, filename, getattr(obj, 'name', None)))

pd.Series(sure_outliers, name="ModelID").to_csv(OUT_DIR / "ccle_sure_outliers.txt", index=False)
pd.Series(possible_outliers, name="ModelID").to_csv(OUT_DIR / "ccle_possible_outliers.txt", index=False)
artifacts_written.extend([OUT_DIR / "ccle_sure_outliers.txt", OUT_DIR / "ccle_possible_outliers.txt"])

if len(meta_baseline_df) > 0:
    meta_baseline_df.to_csv(OUT_DIR / "ccle_metadata_only_validation_baselines.csv", index=False)
    artifacts_written.append(OUT_DIR / "ccle_metadata_only_validation_baselines.csv")

with open(OUT_DIR / "ccle_modeling_schema.json", "w") as f:
    json.dump(schema, f, indent=2)
artifacts_written.append(OUT_DIR / "ccle_modeling_schema.json")

print(f"Wrote {len(artifacts_written)} artifacts to {OUT_DIR.resolve()}")
for p in artifacts_written:
    print(" -", p.name)


Wrote 18 artifacts to /Users/cameroncaputa/PycharmProjects/tp53-rna-prediction/processed
 - ccle_sample_metadata.parquet
 - ccle_X_rna_filtered_all.parquet
 - ccle_X_meta.parquet
 - ccle_gene_filter_qc.parquet
 - ccle_corr_train_filtered_primary.parquet
 - ccle_eda_target_overlap.parquet
 - ccle_outlier_votes.parquet
 - ccle_sample_qc_metrics.parquet
 - ccle_split_membership.parquet
 - ccle_y_primary.parquet
 - ccle_y_all_any.parquet
 - ccle_y_all_nonsyn.parquet
 - ccle_y_all_lof.parquet
 - ccle_mut_type.parquet
 - ccle_sure_outliers.txt
 - ccle_possible_outliers.txt
 - ccle_metadata_only_validation_baselines.csv
 - ccle_modeling_schema.json
